# Module 5: Capstone Challenge

**Course:** UNDP IICPSD Agentic AI & Vibe Coding Training Series 2026 | **Instructor:** Phyo Ko Ko, Senior Cyber Security Analyst

---

## Objective

This capstone combines all 4 modules into one complete end-to-end vibe coding pipeline. You will act as both the engineer and the auditor.

```
[Step 1]              [Step 2]            [Step 3]          [Step 4]
Write a               Retrieve            Scan the    -->   Fix the
structured   -->      context     -->     AI output         vulnerabilities
Markdown prompt       via RAG             for security      and re-scan
(Module 1)            (Module 2)          (Module 3)        (Module 3)
```

## Scenario

Your manager sends you this request:

> *'Build a user profile lookup feature that queries our database by username and returns the user's email and role.'*

Your job is to run this request through the complete vibe coding pipeline below.

---

## Before You Begin

This notebook is **self-contained** - all required functions from Modules 1-4 are provided in the setup cell below. Run it first before attempting any steps.


In [ ]:
# ============================================================
# SETUP: Run this cell first - provides all required functions
# ============================================================
import re
import inspect

# --- From Module 2: RAG Retriever ---
mock_codebase = {
    "auth.py": "def login_user(username, password):\n    print('Logging in user...')\n    return True",
    "db.py": "def connect_database():\n    db_uri = 'postgresql://localhost:5432/main_db'\n    return db_uri",
    "users.py": "def get_user_profile(username):\n    return {'email': username + '@undp.org', 'role': 'Analyst'}",
    "weather.py": "import requests\ndef get_current_weather(city):\n    return f'Weather data for {city}'"
}

def improved_rag_retriever(user_intent):
    print(f"[RAG] Searching codebase for: '{user_intent}'")
    keywords = user_intent.lower().split()
    retrieved_context = ""
    for filename, code in mock_codebase.items():
        if any(kw in filename or kw in code for kw in keywords):
            print(f"  Match: {filename}")
            retrieved_context += f"\n--- {filename} ---\n{code}\n"
    return retrieved_context if retrieved_context else "No relevant context found."

# --- From Module 3: Security Scanner ---
def extended_safety_scanner(target_function):
    print(f"[Scanner] Scanning: {target_function.__name__}")
    source = inspect.getsource(target_function)
    findings = []

    cred_pattern = re.compile(r'(?i)(key|secret|token|password)\s*=\s*[\'"][^\'"]{8,}[\'"]')
    if cred_pattern.findall(source):
        findings.append("CRITICAL - Hardcoded credentials detected")

    if re.search(r'\beval\s*\(|\bexec\s*\(', source):
        findings.append("HIGH - Dangerous function: eval() or exec() detected")

    if re.search(r'f[\'"].*?(SELECT|INSERT|UPDATE|DELETE)', source, re.IGNORECASE):
        findings.append("HIGH - SQL injection risk: raw f-string in SQL query")

    if re.search(r'(?i)DEBUG\s*=\s*True', source):
        findings.append("MEDIUM - Debug flag left on: DEBUG = True")

    print("\n--- Safety Net Audit Report ---")
    if findings:
        for f in findings:
            print(f"  [!] {f}")
        print("\n  Status: BLOCKED - Fix all issues before deployment.")
        return False
    else:
        print("  Status: PASS - No vulnerabilities detected.")
        return True

# --- From Module 4: MCP Server & Agent ---
class MCPServer:
    def __init__(self, name):
        self.name = name
        self.tools = {}
    def register_tool(self, name, description, parameters, handler):
        self.tools[name] = {"name": name, "description": description,
                            "parameters": parameters, "handler": handler}
    def list_tools(self):
        return {name: t["description"] for name, t in self.tools.items()}
    def call_tool(self, tool_name, **kwargs):
        if tool_name not in self.tools:
            return f"Error: Tool '{tool_name}' not found."
        return self.tools[tool_name]["handler"](**kwargs)

def query_database(table, condition):
    mock_db = {"users": [{"id": 1, "name": "Alice", "email": "alice@undp.org", "role": "Admin"},
                         {"id": 2, "name": "Bob", "email": "bob@undp.org", "role": "Analyst"}]}
    results = [row for row in mock_db.get(table, []) if condition.lower() in str(row).lower()]
    return results if results else "No records found."

def read_file(filepath):
    mock_files = {"/logs/access.log": "2026-07-07 10:23 - User alice logged in"}
    return mock_files.get(filepath, f"File '{filepath}' not found.")

def send_notification(channel, message):
    return f"[Notification -> {channel}]: {message}"

server = MCPServer("Capstone-MCP-Server")
server.register_tool("query_database", "Query a database table",
                     {"table": "string", "condition": "string"}, query_database)
server.register_tool("read_file", "Read a file", {"filepath": "string"}, read_file)
server.register_tool("send_notification", "Send a notification",
                     {"channel": "string", "message": "string"}, send_notification)

print("Setup complete. All functions are ready.")
print(f"MCP tools available: {list(server.list_tools().keys())}")


---

## Step 1: Write a Structured Markdown Prompt

Using what you learned in Module 1, write a high-quality Markdown prompt for the scenario.

Your prompt must include:
- A clear **title header** (`#`)
- A **requirements** section with bullet points
- A **constraints** section (security, language, no hardcoded values)
- An **expected output** example in a code block


In [ ]:
my_prompt = """
# TODO: Write your structured Markdown prompt here for the user profile lookup feature
"""

print("=== Step 1: Your Structured Prompt ===")
print(my_prompt)


---

## Step 2: Retrieve Relevant Context via RAG

Using what you learned in Module 2, run your prompt through the RAG retriever to find which files in the codebase are relevant to this task.

Write a specific query that captures the intent: looking up a user profile from a database.


In [ ]:
print("=== Step 2: RAG Context Retrieval ===")

# TODO: Replace with a query relevant to the user profile lookup scenario
retrieved = improved_rag_retriever("your query here")
print(retrieved)


---

## Step 3: Scan the AI-Generated Code

The AI agent responded to your prompt with the function below. Before using it, run it through the Safety Net scanner from Module 3.

Identify all the vulnerabilities before looking at the scanner output.


In [ ]:
def ai_generated_profile_lookup():
    db_secret = "prod_db_pass_9x!2Z"
    DEBUG = True
    username = input("Enter username: ")
    query = f"SELECT email, role FROM users WHERE username = '{username}'"
    result = eval(query)
    return result

print("=== Step 3: Scan the AI-Generated Code ===")
extended_safety_scanner(ai_generated_profile_lookup)


---

## Step 4: Fix the Vulnerabilities & Re-scan

Rewrite `ai_generated_profile_lookup` as a secure version:

- Remove the hardcoded `db_secret` (use a comment like `# loaded from os.environ` instead)
- Remove `DEBUG = True`
- Remove `eval()` - use a safe mock lookup instead
- Remove the raw SQL f-string - use a parameterized approach

The scanner should return **PASS** when you run it on your fixed version.


In [ ]:
# TODO: Write the secure fixed version here
# Tip: load secrets from your .env file (copy .env.example -> .env and fill in your values)
# from dotenv import load_dotenv; load_dotenv()

def fixed_profile_lookup():
    # db_connection = os.environ.get("DB_URI")  # loaded from .env, not hardcoded
    # Use parameterized queries and safe lookups instead of f-strings and eval()
    pass

print("=== Step 4: Scan Your Fixed Code ===")
extended_safety_scanner(fixed_profile_lookup)

---

## Capstone Reflection

**Q1.** What was the most important thing the structured Markdown prompt helped you clarify before writing any code?

> *Your answer here...*

**Q2.** Which files did the RAG retriever surface in Step 2? Were they the right ones? If not, how would you improve your query?

> *Your answer here...*

**Q3.** The AI-generated code had 3 vulnerabilities. In a real production team using vibe coding at scale, what process or tooling would you put in place to prevent these from reaching deployment?

> *Your answer here...*

---

*End of Capstone Challenge | UNDP IICPSD Technical Training Series 2026*
